In [1]:
import pandas as pd
import numpy as np
import warnings
from sklearn.exceptions import DataConversionWarning
from sklearn.preprocessing import RobustScaler
from dowhy import CausalModel
from IPython.display import display
from scipy.stats import spearmanr


warnings.filterwarnings("ignore", category=DataConversionWarning)

In [2]:
df = pd.read_csv("alerts_with_category_with_apk_size_updated.csv")
print("Original rows:", len(df))

df = df[~df["code_location"].str.contains("obfuscated", case=False, na=False)].copy()
print("After removing obfuscated:", len(df))

df["verdict"] = df["verdict"].astype(int)

download_order = ['<100','100-500','500-1k','1k-5k','5k-10k','10k-50k',
                  '50k-100k','100k-500k','500k-1M','1M-5M','>5M']
ord_map = {c:i for i,c in enumerate(download_order)}
df["app_popularity_encoded"] = df["app_popularity"].map(ord_map)
df = df.dropna(subset=["app_popularity_encoded"]).copy()
df["app_popularity_encoded"] = df["app_popularity_encoded"].astype(int)

scaler = RobustScaler()
df["apk_size_scaled"] = scaler.fit_transform(df[["apk_size"]])


Original rows: 6140
After removing obfuscated: 5612


In [3]:
small_libs = ["SocialMedia", "Analytics", "Cloud"]
df["lib_grouped"] = df["code_location"].replace(small_libs, "Small_Libraries")
df.loc[df["lib_grouped"] == "developer_written", "lib_grouped"] = "App_Source_Code"

print("\nGrouped library counts:")
print(df["lib_grouped"].value_counts())

baseline_name = "App_Source_Code"
libs = [baseline_name] + [l for l in df["lib_grouped"].unique() if l != baseline_name]
print("\nLibrary levels:")
for l in libs:
    print("-", l)


Grouped library counts:
lib_grouped
Utilities          3039
others             1341
App_Source_Code     511
Android             494
Small_Libraries     227
Name: count, dtype: int64

Library levels:
- App_Source_Code
- others
- Android
- Utilities
- Small_Libraries


In [4]:
refutation_methods = [
    "random_common_cause",
    "placebo_treatment_refuter",
    "data_subset_refuter",
]

In [5]:
def make_pairwise_balanced(df_in,lib_name,baseline="App_Source_Code",random_state=100):
    df_sub = df_in[df_in["lib_grouped"].isin([baseline, lib_name])].copy()

    base = df_sub[df_sub["lib_grouped"] == baseline]
    lib  = df_sub[df_sub["lib_grouped"] == lib_name]

    n_base = len(base)
    n_lib  = len(lib)

    base_bal = base.copy()

    if n_lib >= n_base:
        lib_bal = lib.sample(n=n_base, replace=False, random_state=random_state)
    else:
        lib_bal = lib.sample(n=n_base, replace=True, random_state=random_state)

    df_bal = pd.concat([base_bal, lib_bal], axis=0)
    df_bal = df_bal.sample(frac=1, random_state=random_state).reset_index(drop=True)
   
    return df_bal



In [6]:
def apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate):
    ref_rows = []
    for method in refutation_methods:
        if method == "placebo_treatment_refuter":
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method, placebo_type="permute", num_simulations=200)
        elif method == "data_subset_refuter":
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method, subset_fraction=0.8, num_simulations=200)
        else:
            ref = model.refute_estimate(identified_estimand, estimate, method_name=method)

        print(f"\nRefuter: {method}\n", ref)

        ref_rows.append({
            "library": lib_name,
            "setting": label,
            "refuter": method,
            "orig_effect": ate,
            "new_effect": getattr(ref, "new_effect", None),
            "p_value": getattr(ref, "p_value", None)
        })
        
    return ref_rows

In [7]:
def correlation_test(library, baseline, df):
    rho, p = spearmanr( df["combined"], df["verdict"])
    print(f"{library} -> {baseline} (Spearman)")
    print(f"rho = {rho:.4f}, p = {p:.4f}")


In [8]:
def run_library_contrast(df_sub, lib_name, baseline="App_Source_Code", label="imbalanced"):
    df_sub = df_sub.copy()
    df_sub["treatment_lib"] = (df_sub["lib_grouped"] == lib_name).astype(int)

    print("\n" + "-"*14 + f" {lib_name} vs {baseline} " + "-"*14)
    # print("Counts:", {baseline: n0, lib_name: n1})


    df_all = df_sub.copy()
    df_all["combined"] = 1
    df_baseline = df_sub[df_sub["treatment_lib"] == 0].copy()
    df_baseline["combined"] = 0
    df_final = pd.concat([df_all, df_baseline], ignore_index=True)
    df_sub = df_final.copy()

    correlation_test(lib_name, baseline, df_sub)

    print(df_sub["combined"].value_counts())
    n0 = int((df_sub["combined"] == 0).sum())
    n1 = int((df_sub["combined"] == 1).sum())


    causal_graph = """
    digraph {
        combined -> verdict;
        app_popularity_encoded -> combined;
        app_popularity_encoded -> verdict;
        apk_size_scaled -> combined;
        apk_size_scaled -> verdict;
        app_popularity_encoded -> apk_size_scaled;
    }
    """

    model = CausalModel(
        data=df_sub,
        treatment="combined",
        outcome="verdict",
        graph=causal_graph.replace("\n", " "),
        common_causes=["app_popularity_encoded","apk_size_scaled"]
    )

    identified_estimand = model.identify_effect(proceed_when_unidentifiable=True)

    estimate = model.estimate_effect(
        identified_estimand,
        method_name="backdoor.propensity_score_matching",
        target_units="ate",
        confidence_intervals='bootstrap',
        method_params={
            "num_simulations": 300,
            "sample_size_fraction": 1.0,
            "confidence_level": 0.95,
        },
    )

    ate = float(estimate.value)
    try:
        ci_low, ci_high = estimate.get_confidence_intervals()
        ci_low, ci_high = float(ci_low), float(ci_high)
    except Exception:
        ci_low, ci_high = None, None

    print("ATE:", ate)
    if ci_low is not None:
        print("95% CI:", (ci_low, ci_high))


    ref_rows = apply_refutation_tests(lib_name, label, model, identified_estimand, estimate, ate)

    result_row = {
        "library": lib_name,
        "setting": label,
        "n_baseline": n0,
        "n_treatment": n1,
        "ATE": ate,
        "CI_low": ci_low,
        "CI_high": ci_high
    }

    return result_row, ref_rows




In [9]:
main_rows = []
ref_rows_all = []

for lib in libs:
    if lib == baseline_name:
        continue

    df_bal = make_pairwise_balanced(df, lib, baseline=baseline_name, random_state=100)
    main_row_b, ref_b = run_library_contrast(df_bal, lib, baseline=baseline_name, label="balanced")
    main_rows.append(main_row_b)
    ref_rows_all.extend(ref_b)



-------------- others vs App_Source_Code --------------
others -> App_Source_Code (Spearman)
rho = -0.0381, p = 0.1357
combined
1    1022
0     511
Name: count, dtype: int64


ATE: 0.060013046314416174
95% CI: (0.08871493803000652, 0.2609262883235486)

Refuter: random_common_cause
 Refute: Add a random common cause
Estimated effect:0.060013046314416174
New effect:0.06001304631441617
p value:1.0


Refuter: placebo_treatment_refuter
 Refute: Use a Placebo Treatment
Estimated effect:0.060013046314416174
New effect:0.004132420091324201
p value:0.94


Refuter: data_subset_refuter
 Refute: Use a subset of data
Estimated effect:0.060013046314416174
New effect:-0.06604404567699836
p value:0.01


-------------- Android vs App_Source_Code --------------
Android -> App_Source_Code (Spearman)
rho = 0.0537, p = 0.0355
combined
1    1022
0     511
Name: count, dtype: int64
ATE: 0.0319634703196347
95% CI: (-0.06523157208088715, 0.12720156555772993)

Refuter: random_common_cause
 Refute: Add a random common cause
Estimated effect:0.0319634703196347
New effect:0.031963470319634694
p value:1.0


Refuter: placebo_treatment_refuter
 Refute: Use a Placebo Treatment
Estimated eff

In [10]:
main_df = pd.DataFrame(main_rows).sort_values(["library", "setting"])
ref_df = pd.DataFrame(ref_rows_all)

print("\n\n","-"*25 + "Detailed (ATEs) " + "-"*25)
print(main_df[["library","setting","n_baseline","n_treatment","ATE","CI_low","CI_high"]])

pivot = main_df.pivot(index="library", columns="setting", values="ATE")
print("\n\n","-"*25 + "Summary ATE " + "-"*25)
print(pivot)



 -------------------------Detailed (ATEs) -------------------------
           library   setting  n_baseline  n_treatment       ATE    CI_low  \
1          Android  balanced         511         1022  0.031963 -0.065232   
3  Small_Libraries  balanced         511         1022  0.058708 -0.070450   
2        Utilities  balanced         511         1022 -0.182648 -0.288976   
0           others  balanced         511         1022  0.060013  0.088715   

    CI_high  
1  0.127202  
3  0.112851  
2 -0.116765  
0  0.260926  


 -------------------------Summary ATE -------------------------
setting          balanced
library                  
Android          0.031963
Small_Libraries  0.058708
Utilities       -0.182648
others           0.060013
